# 实验三：分词算法的评测

为了进行计算，我们首先需要将分词结果（如 "商品 和 服务"）转换为标准的、与位置无关的区间格式（如 {(0, 2), (3, 5)}），这样就可以方便地用集合运算来比较两个分词结果的差异。

In [10]:
import re
import os

# --- 预先定义实验一中的函数和变量，确保本实验能独立运行 ---
def load_dictionary(dict_file):
    with open(dict_file, encoding="utf-8") as f:
        return {line.strip().split("\t")[0] for line in f}

def forward_segment(text, dic):
    word_list = []
    i = 0
    while i < len(text):
        longest_word = text[i]
        for j in range(i + 1, len(text) + 1):
            word = text[i:j]
            if word in dic:
                if len(word) > len(longest_word):
                    longest_word = word
        word_list.append(longest_word)
        i += len(longest_word)
    return word_list

def backward_segment(text, dic):
    word_list = []
    i = len(text) - 1
    while i >= 0:
        longest_word = text[i]
        for j in range(0, i):
            word = text[j: i + 1]
            if word in dic:
                if len(word) > len(longest_word):
                    longest_word = word
                    break
        word_list.insert(0, longest_word)
        i -= len(longest_word)
    return word_list

def count_single_char(word_list: list):
    return sum(1 for word in word_list if len(word) == 1)

def bidirectional_segment(text, dic):
    f = forward_segment(text, dic)
    b = backward_segment(text, dic)
    if len(f) < len(b): return f
    elif len(f) > len(b): return b
    else:
        if count_single_char(f) < count_single_char(b): return f
        else: return b
# --- 预定义结束 ---


def to_region(segmentation: str) -> set:
    """将以空格分隔的分词结果字符串转换为词语区间的集合。"""
    region = set()
    start = 0
    for word in re.compile("\\s+").split(segmentation.strip()):
        end = start + len(word)
        region.add((start, end))
        start = end
    return region

def prf(gold: str, pred: str, dic) -> tuple:
    """
    计算P, R, F1, OOV-R, IV-R。
    :param gold: 标准答案文件路径。
    :param pred: 预测结果文件路径。
    :param dic: 词典。
    :return: (P, R, F1, OOV_R, IV_R)
    """
    A_size, B_size, A_cap_B_size = 0, 0, 0  # A:标准答案词数, B:预测结果词数, A&B:正确词数
    OOV, IV = 0, 0  # OOV:标准答案中的未登录词数, IV:标准答案中的登录词数
    OOV_R, IV_R = 0, 0 # OOV_R:正确切分的OOV数, IV_R:正确切分的IV数
    
    with open(gold, encoding="utf-8") as gd, open(pred, encoding="utf-8") as pd:
        for g, p in zip(gd, pd):
            A = to_region(g)
            B = to_region(p)
            A_cap_B = A & B
            
            A_size += len(A)
            B_size += len(B)
            A_cap_B_size += len(A_cap_B)
            
            # 统计OOV和IV的召回情况
            text = re.sub("\\s+", "", g.strip())
            for start, end in A:
                word = text[start:end]
                if word in dic:
                    IV += 1
                    if (start, end) in A_cap_B:
                        IV_R += 1
                else:
                    OOV += 1
                    if (start, end) in A_cap_B:
                        OOV_R += 1
    
    p = A_cap_B_size / B_size * 100 if B_size > 0 else 0
    r = A_cap_B_size / A_size * 100 if A_size > 0 else 0
    f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
    oov_r = OOV_R / OOV * 100 if OOV > 0 else 0
    iv_r = IV_R / IV * 100 if IV > 0 else 0
    
    return p, r, f1, oov_r, iv_r

# 测试 to_region 函数
print(f"'商品 和 服务' 的区间表示: {to_region('商品 和 服务')}")

'商品 和 服务' 的区间表示: {(2, 3), (0, 2), (3, 5)}


In [8]:
import re

def to_region(segmentation: str) -> set:
    """将以空格分隔的分词结果字符串转换为词语区间的集合。"""
    region = set()
    start = 0
    # 分割后得到的列表是：['商品', '和', '服务']（顺序固定）
    for word in re.compile("\\s+").split(segmentation.strip()):
        end = start + len(word)
        print(f"插入集合的元素：({start}, {end})")  # 新增打印，看插入顺序
        region.add((start, end))
        start = end
    return region

print(f"'商品 和 服务' 的区间表示: {to_region('商品 和 服务')}")

插入集合的元素：(0, 2)
插入集合的元素：(2, 3)
插入集合的元素：(3, 5)
'商品 和 服务' 的区间表示: {(2, 3), (0, 2), (3, 5)}


为什么调用 to_region('商品 和 服务') 输出的集合是 {(2, 3), (0, 2), (3, 5)}，而非按插入顺序排列的 {(0, 2),(2, 3), (3, 5)}，核心原因是Python 的集合（set）是无序容器，不会保留元素的插入顺序。

现在，我们定义一个自动化评测流程的函数，它接收一个分词算法，用它来处理测试集，然后调用prf函数给出最终的评测报告。

In [12]:
# 定义评测所需文件路径 (请确保路径正确)
sighan05 = "./corpus_data/第二届国际中文分词评测/icwb2-data/"
msr_dict_path = os.path.join(sighan05, 'gold', 'msr_training_words.utf8')
msr_test_path = os.path.join(sighan05, 'testing', 'msr_test.utf8')
msr_output_path = os.path.join(sighan05, 'testing', 'msr_output.txt') # 临时输出文件
msr_gold_path = os.path.join(sighan05, 'gold', 'msr_test_gold.utf8')

def process_and_evaluate(gold_path, test_path, output_path, dic, segment_function):
    """自动化处理测试集并进行评测的流程函数"""
    print(f"正在使用 {segment_function.__name__} 处理测试集...")
    with open(test_path, encoding="utf-8") as test_file, open(output_path, 'w', encoding="utf-8") as output_file:
        for line in test_file:
            # 将分词结果用双空格连接，写入输出文件
            output_file.write("  ".join(segment_function(line.strip(), dic)))
            output_file.write("\n")
    print("处理完成，开始评测...")
    
    # 调用评测函数
    p, r, f1, oov_r, iv_r = prf(gold_path, output_path, dic)
    print(f"P: {p:.2f}%, R: {r:.2f}%, F1: {f1:.2f}%, OOV-R: {oov_r:.2f}%, IV-R: {iv_r:.2f}%")

# 加载评测所需的MSR词典
msr_word_dict = load_dictionary(msr_dict_path)

# --- 依次对三种算法进行评测 ---
print("--- 前向最大匹配 (FMM) 评测 ---")
process_and_evaluate(msr_gold_path, msr_test_path, msr_output_path, msr_word_dict, forward_segment)

print("\n--- 逆向最大匹配 (BMM) 评测 ---")
process_and_evaluate(msr_gold_path, msr_test_path, msr_output_path, msr_word_dict, backward_segment)

print("\n--- 双向最大匹配 (Bi-MM) 评测 ---")
process_and_evaluate(msr_gold_path, msr_test_path, msr_output_path, msr_word_dict, bidirectional_segment)

--- 前向最大匹配 (FMM) 评测 ---
正在使用 forward_segment 处理测试集...
处理完成，开始评测...
P: 91.62%, R: 95.57%, F1: 93.55%, OOV-R: 2.47%, IV-R: 98.10%

--- 逆向最大匹配 (BMM) 评测 ---
正在使用 backward_segment 处理测试集...
处理完成，开始评测...
P: 91.43%, R: 95.38%, F1: 93.36%, OOV-R: 2.47%, IV-R: 97.90%

--- 双向最大匹配 (Bi-MM) 评测 ---
正在使用 bidirectional_segment 处理测试集...
处理完成，开始评测...
P: 91.63%, R: 95.52%, F1: 93.53%, OOV-R: 2.47%, IV-R: 98.05%



1. 效果对比：
- 三种算法的 F1 值分别为：
    - FMM: 93.55%
    - BMM: 93.36%
    - Bi-MM: 93.53%
- 因此，本次评测中 FMM 的 F1 值最高，效果最好。
- 这说明本次实验结果并没有明显验证“双向匹配一定更优”的猜想。Bi-MM 的表现非常接近 FMM，但并不是最优。
- 更合理的结论是：双向匹配通常是一种有效改进策略，但它不是在所有数据集、所有词典条件下都绝对最优。

2. 瓶颈分析：
- OOV-R 只有 2.47%，说明对未登录词的识别能力非常差。
- 原因是这三种方法本质上都是“基于词典匹配”的算法，分词时主要依赖词典中是否存在某个词。
- 一旦测试文本中出现词典里没有的新词、专有名词、地名、人名或新造词，算法就无法正确整体识别它们，往往只能把它们切成单字或切成词典中已有的小词。
- 这揭示了词典匹配方法的根本缺陷：严重依赖词典覆盖率，缺乏发现未登录词的能力。

结论：
- 词典匹配算法对登录词（IV）效果较好，所以 IV-R 很高。
- 但对未登录词（OOV）几乎无能为力，这正是所有基于词典匹配的分词方法都难以回避的根本问题。
